In [ ]:
# переименуем столбец cast в actors, используя sql. Если переименовывать через sqlite, нужно будет создавать новую таблицу
ALTER TABLE netflix_titles
RENAME COLUMN cast TO actors;

In [ ]:
#создадим первую таблицу для фильмов
import sqlite3

con = sqlite3.connect('task1.sqlite')
cur = con.cursor()

with con:
    cur.execute("""
        CREATE TABLE movie (
            movie_id INT NOT NULL PRIMARY KEY,
            title TEXT
        );
    """)

con.close()

In [ ]:
#создадим вторую таблицу для актеров
import sqlite3

con = sqlite3.connect('task1.sqlite')
cur = con.cursor()

with con:
    cur.execute("""
        CREATE TABLE actors (
            actor_id INT NOT NULL PRIMARY KEY,
            name TEXT
        );
    """)

con.close()

In [ ]:
#создадим третью таблицу для связи фильма и актера
import sqlite3

con = sqlite3.connect('task1.sqlite')
cur = con.cursor()

with con:
    cur.execute("""
        CREATE TABLE movie_actors (
            movie_id INT,
            actor_id INT,
            PRIMARY KEY (movie_id, actor_id),
            FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
            FOREIGN KEY (actor_id) REFERENCES actors(actor_id)
        );
    """)

con.close()

In [ ]:
#открываем дб netflix, переносим данные о id и названии фильмов в нашу таблицу movie
import sqlite3

con = sqlite3.connect('task1.sqlite')
connetflix = sqlite3.connect('netflix.sqlite')
cur = con.cursor()
curnetflix = connetflix.cursor()

with connetflix:
    with con:
        curnetflix.execute("SELECT show_id, title FROM netflix_titles WHERE type = 'Movie'")
        res = curnetflix.fetchall()
        cur.executemany("INSERT INTO movie (movie_id, title) VALUES (?, ?)", res)

con.close()
connetflix.close()

In [ ]:
#переносим данные о актерах, нумеруя id с 1
import sqlite3
a = 0

def up():
    global a
    a += 1
    return a

con = sqlite3.connect('task1.sqlite')
connetflix = sqlite3.connect('netflix.sqlite')
cur = con.cursor()
curnetflix = connetflix.cursor()

actors = []

with connetflix:
    with con:
        curnetflix.execute("SELECT actors FROM netflix_titles")
        res = curnetflix.fetchall()
        for row in res:
            temp = row[0].split(", ")
            for item in temp:
                actors.append(item)
            actors = list(set(actors[1:])) # первый символ пустота
        cur.executemany("INSERT INTO actors (actor_id, name) VALUES (?, ?)",
                                   [(up(), actor) for actor in actors])


con.close()
connetflix.close()

In [ ]:
#заполняем третью таблицу movie_id actor_id
import sqlite3

con = sqlite3.connect('task1.sqlite')
connetflix = sqlite3.connect('netflix.sqlite')
cur = con.cursor()
curnetflix = connetflix.cursor()

movie_actor = []

with connetflix:
    with con:
        curnetflix.execute("SELECT show_id, actors FROM netflix_titles")
        res = curnetflix.fetchall()
        for row in res:
            film_id = row[0]
            temp = row[1].split(", ")
            for item in temp:
                movie_actor.append((film_id, item))
        movie_actor = list(set(movie_actor))
        for item in movie_actor:
            cur.execute("SELECT actor_id FROM actors WHERE name = ?", (item[1],))
            res = cur.fetchone()
            if res is not None:
                actor_id = res[0]
                cur.execute("INSERT INTO movie_actors (movie_id, actor_id) VALUES (?, ?)", (item[0], actor_id))

con.close()
connetflix.close()

In [ ]:
#выбираем двух наиболее часто снимающихся вместе актеров
import sqlite3

con = sqlite3.connect('task1.sqlite')
cur = con.cursor()


with con:
    cur.execute("""
    SELECT
    a1.actor_id AS actor_id1,
    a2.actor_id AS actor_id2,
    COUNT(*) AS frequency
    FROM movie_actors a1
    JOIN movie_actors a2 ON a1.movie_id = a2.movie_id AND a1.actor_id < a2.actor_id
    GROUP BY a1.actor_id, a2.actor_id
    ORDER BY frequency DESC
    LIMIT 1;   
    """)
    res = cur.fetchone()
    cur.execute("SELECT name FROM actors WHERE actor_id = ?", (res[0],))
    name1 = cur.fetchone()
    cur.execute("SELECT name FROM actors WHERE actor_id = ?", (res[1],))
    name2 = cur.fetchone()
    print("Наиболее часто снимающиеся вместе актеры: {}, {}. \nОни снялись вместе в {} фильмах.".
          format(name1[0], name2[0], res[2]))

con.close()